# Retail Sales Data Pipeline

**Author: Feras Elghandour**

Clean heterogeneous retail transaction fields and produce repeatable pricing and revenue summaries.

Reusable type-cleaning and imputation functions, date and product-feature derivation, grouped exploratory analysis, and pricing/revenue visualizations.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Record versions for reproducibility
print(f"numpy:      {np.__version__}")
print(f"pandas:     {pd.__version__}")
print(f"matplotlib: {plt.matplotlib.__version__}")
print(f"seaborn:    {sns.__version__}")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 6)})

In [ ]:
df = pd.read_csv("laptop_sales.csv")
df.head(10)

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
def clean_types_and_missing(df):
    """
    Convert columns to correct types, parse dates, and impute missing
    numeric values with column medians.
    
    Parameters
    ----------
    df : pd.DataFrame
        Raw laptop sales DataFrame.
    
    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with proper types and no missing numerics.
    """
    df = df.copy()
    
    # Parse sale_date to datetime
    df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
    
    # Ensure numeric columns are truly numeric
    numeric_cols = [
        "base_price", "final_price", "discount_pct", "quantity_sold",
        "revenue", "customer_rating", "weight_kg", "battery_life_hours",
        "ram_gb", "storage_gb", "cpu_cores", "cpu_base_ghz", "display_inches",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    
    # Impute missing numerics with median
    missing_before = df[numeric_cols].isnull().sum()
    for col in numeric_cols:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())
    
    missing_after = df[numeric_cols].isnull().sum()
    print("Missing values BEFORE cleaning:")
    print(missing_before[missing_before > 0])
    print("\nMissing values AFTER cleaning:")
    print(missing_after[missing_after > 0] if missing_after.sum() > 0 else "None — all imputed.")
    
    return df

In [ ]:
def add_derived_features(df):
    """
    Engineer new columns that support downstream analysis and ML models.
    
    Creates:
    - sale_month, sale_quarter, day_of_week: temporal features from sale_date
    - has_dedicated_gpu: binary flag (1 = discrete GPU, 0 = integrated)
    - discount_amount: absolute dollar savings (base_price - final_price)
    - spec_score: composite hardware score (0-1) combining RAM, storage, and CPU cores
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with cleaned numeric columns and parsed dates.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with additional derived columns.
    """
    df = df.copy()
    
    # Temporal features
    df["sale_month"]   = df["sale_date"].dt.to_period("M").astype(str)
    df["sale_quarter"] = df["sale_date"].dt.to_period("Q").astype(str)
    df["day_of_week"]  = df["sale_date"].dt.day_name()
    
    # GPU flag
    df["has_dedicated_gpu"] = (
        ~df["gpu"].isin(["Integrated", "Apple GPU (integrated)"])
    ).astype(int)
    
    # Discount amount
    df["discount_amount"] = (df["base_price"] - df["final_price"]).round(2)
    
    # Composite spec score (normalized 0-1)
    df["spec_score"] = (
        df["ram_gb"]     / df["ram_gb"].max()     * 0.4 +
        df["storage_gb"] / df["storage_gb"].max() * 0.3 +
        df["cpu_cores"]  / df["cpu_cores"].max()  * 0.3
    ).round(3)
    
    print(f"Added {6} derived columns. New shape: {df.shape}")
    return df

In [ ]:
# Apply cleaning pipeline
df = clean_types_and_missing(df)
df = add_derived_features(df)
df.head()

In [ ]:
def exploratory_analysis(df, group_col="brand", value_col="final_price"):
    """
    Perform exploratory analysis on the dataset.
    
    Outputs:
    1. Descriptive statistics for key numeric columns.
    2. Group-level aggregation (mean, median, count) by a categorical column.
    3. Correlation matrix for numeric features.
    
    Parameters
    ----------
    df : pd.DataFrame
        Cleaned and feature-engineered DataFrame.
    group_col : str
        Categorical column to group by (default: 'brand').
    value_col : str
        Numeric column to aggregate (default: 'final_price').
    
    Returns
    -------
    tuple
        (descriptive_stats, group_summary, correlation_matrix)
    """
    # 1. Descriptive statistics
    desc = df[["final_price", "ram_gb", "storage_gb", "cpu_cores",
               "customer_rating", "battery_life_hours", "quantity_sold"]].describe().round(2)
    print("=== Descriptive Statistics ===")
    print(desc)
    
    # 2. Grouped summary
    group_summary = (
        df.groupby(group_col)[value_col]
        .agg(["mean", "median", "std", "count"])
        .round(2)
        .sort_values("mean", ascending=False)
    )
    print(f"\n=== {value_col} by {group_col} ===")
    print(group_summary)
    
    # 3. Correlation matrix
    num_cols = ["final_price", "ram_gb", "storage_gb", "cpu_cores",
                "cpu_base_ghz", "discount_pct", "quantity_sold",
                "customer_rating", "spec_score"]
    corr = df[num_cols].corr().round(3)
    print("\n=== Correlation Matrix (selected features) ===")
    print(corr)
    
    return desc, group_summary, corr

In [ ]:
desc_stats, brand_summary, corr_matrix = exploratory_analysis(df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["final_price"], kde=True, bins=40, color="#20808D", ax=ax)
ax.set_title("Distribution of Final Laptop Prices", fontsize=14, fontweight="bold")
ax.set_xlabel("Final Price (USD)")
ax.set_ylabel("Count")
ax.axvline(df["final_price"].median(), color="#A84B2F", linestyle="--",
           label=f'Median: ${df["final_price"].median():,.0f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
brand_gpu = (
    df.groupby(["brand", "has_dedicated_gpu"])["final_price"]
    .mean()
    .unstack(fill_value=0)
    .rename(columns={0: "Integrated GPU", 1: "Dedicated GPU"})
    .sort_values("Dedicated GPU", ascending=False)
)
brand_gpu.plot(kind="bar", ax=ax, color=["#20808D", "#A84B2F"], edgecolor="white")
ax.set_title("Average Laptop Price by Brand and GPU Type", fontsize=14, fontweight="bold")
ax.set_xlabel("Brand")
ax.set_ylabel("Average Final Price (USD)")
ax.legend(title="GPU Type")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
monthly_rev = (
    df.groupby("sale_month")["revenue"]
    .sum()
    .reset_index()
    .sort_values("sale_month")
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(range(len(monthly_rev)), monthly_rev["revenue"], alpha=0.15, color="#20808D")
ax.plot(range(len(monthly_rev)), monthly_rev["revenue"], marker="o",
        color="#20808D", linewidth=2, markersize=5)
ax.set_xticks(range(0, len(monthly_rev), 2))
ax.set_xticklabels(monthly_rev["sale_month"].iloc[::2], rotation=45, ha="right")
ax.set_title("Monthly Revenue Trend (2023–2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Total Revenue (USD)")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
plt.show()